# SENTINEL Model Training Demo Notebook

This notebook documents how we trained and compared different ML approaches for the DNS IDS model used by SENTINEL.

It is designed as a demo + audit artifact, not just a one-shot training script.

## 1) Goals

We want a model that:
- catches malicious DNS behavior with high recall
- keeps false positives manageable for analysts
- remains explainable enough to pair with rule/behavior evidence

This notebook demonstrates:
1. multiple model candidates
2. class imbalance handling
3. cross-validation + holdout testing
4. threshold tuning
5. feature-group ablation

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import matplotlib.pyplot as plt

RANDOM_STATE = 42
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = ROOT / 'data' / 'features' / 'synthetic_training_data.csv'
DATA_PATH


In [ ]:
from models.train import FEATURE_COLS

assert DATA_PATH.exists(), f'Missing dataset: {DATA_PATH}'
df = pd.read_csv(DATA_PATH).fillna(0)

for c in FEATURE_COLS:
    if c not in df.columns:
        df[c] = 0.0

assert 'label' in df.columns, 'Dataset must contain label column'

X = df[FEATURE_COLS].astype(float)
y = df['label'].astype(int)

print('Dataset shape:', df.shape)
print('Class distribution:')
print(y.value_counts())
print('Attack ratio:', round(y.mean(), 4))


## 2) Candidate Models (Different Tries)

Approaches we evaluate:
- **Logistic Regression** (simple linear baseline)
- **Random Forest** (robust non-linear baseline)
- **Gradient Boosting** (fallback boosting model)
- **XGBoost** when available (production-leaning boosted trees)

We evaluate each with:
- holdout F1
- 5-fold CV F1 mean/std

In [ ]:
models = {}

models['logreg'] = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))
])

models['random_forest'] = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

models['gradient_boosting'] = GradientBoostingClassifier(random_state=RANDOM_STATE)

try:
    from xgboost import XGBClassifier
    models['xgboost'] = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    print('XGBoost available: yes')
except Exception as e:
    print('XGBoost available: no ->', e)

list(models.keys())


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
fitted = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    holdout_f1 = f1_score(y_test, y_pred)

    cv_scores = cross_val_score(model, X, y, cv=cv, scoring='f1')

    rows.append({
        'model': name,
        'holdout_f1': holdout_f1,
        'cv_f1_mean': cv_scores.mean(),
        'cv_f1_std': cv_scores.std(),
    })
    fitted[name] = model

results = pd.DataFrame(rows).sort_values(['holdout_f1', 'cv_f1_mean'], ascending=False).reset_index(drop=True)
results


In [ ]:
best_name = results.iloc[0]['model']
best_model = fitted[best_name]
print('Best model:', best_name)

y_best = best_model.predict(X_test)
print(classification_report(y_test, y_best))
print('Confusion matrix:\n', confusion_matrix(y_test, y_best))


## 3) Imbalance Handling Try (SMOTE)

This compares baseline training vs SMOTE-resampled training for one strong candidate (RandomForest).

If `imbalanced-learn` is missing, this cell will skip gracefully.

In [ ]:
from sklearn.base import clone

rf_base = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf_base.fit(X_train, y_train)
base_pred = rf_base.predict(X_test)
base_f1 = f1_score(y_test, base_pred)

smote_f1 = None
try:
    from imblearn.over_sampling import SMOTE
    X_res, y_res = SMOTE(random_state=RANDOM_STATE).fit_resample(X_train, y_train)
    rf_smote = clone(rf_base)
    rf_smote.fit(X_res, y_res)
    smote_pred = rf_smote.predict(X_test)
    smote_f1 = f1_score(y_test, smote_pred)
except Exception as e:
    print('SMOTE step skipped:', e)

print({'rf_base_f1': base_f1, 'rf_smote_f1': smote_f1})


## 4) Threshold Tuning (Probability Models)

Security products often need a threshold beyond 0.5.
SENTINEL currently uses 0.75 in runtime for malicious flagging.

This section sweeps thresholds and shows precision/recall/F1 tradeoffs.

In [ ]:
if hasattr(best_model, 'predict_proba'):
    probs = best_model.predict_proba(X_test)[:, 1]
else:
    # fallback for models without predict_proba
    raw = best_model.predict(X_test)
    probs = np.asarray(raw, dtype=float)

threshold_rows = []
for th in [0.50, 0.60, 0.70, 0.75, 0.80, 0.90]:
    y_hat = (probs >= th).astype(int)
    threshold_rows.append({
        'threshold': th,
        'precision': precision_score(y_test, y_hat, zero_division=0),
        'recall': recall_score(y_test, y_hat, zero_division=0),
        'f1': f1_score(y_test, y_hat, zero_division=0),
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(threshold_df['threshold'], threshold_df['precision'], marker='o', label='precision')
plt.plot(threshold_df['threshold'], threshold_df['recall'], marker='o', label='recall')
plt.plot(threshold_df['threshold'], threshold_df['f1'], marker='o', label='f1')
plt.title('Threshold Sweep')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.ylim(0,1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## 5) Feature Group Ablation (Approach Comparison)

We test whether lexical-only vs lexical+temporal vs full feature set materially changes F1.

This helps explain why the hybrid feature design exists.

In [ ]:
lexical_cols = [
    'query_length','subdomain_length','label_count','max_label_length','full_entropy','subdomain_entropy',
    'digit_ratio','hyphen_count','consonant_vowel_ratio','longest_consonant_run','hex_ratio','looks_base64'
]
temporal_cols = ['is_beaconing','beacon_score','iat_cv','query_rate_per_min']
session_cols = ['unique_subdomains','nxdomain_rate','txt_query_ratio']

ablation_sets = {
    'lexical_only': lexical_cols,
    'lexical_temporal': lexical_cols + temporal_cols,
    'full_set': FEATURE_COLS,
}

ablation_rows = []
for name, cols in ablation_sets.items():
    X_sub = df[cols].astype(float)
    model = RandomForestClassifier(n_estimators=250, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
    scores = cross_val_score(model, X_sub, y, cv=cv, scoring='f1')
    ablation_rows.append({'feature_set': name, 'cv_f1_mean': scores.mean(), 'cv_f1_std': scores.std()})

pd.DataFrame(ablation_rows).sort_values('cv_f1_mean', ascending=False)


## 6) Save Best Model Artifact (Optional)

If you want to export the best model from this notebook run, enable `SAVE=True`.

By default we keep this disabled to avoid accidental overwrite.

In [ ]:
SAVE = False
if SAVE:
    import joblib
    out_dir = ROOT / 'models' / 'saved'
    out_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(best_model, out_dir / 'dns_classifier.pkl')
    print('Saved:', out_dir / 'dns_classifier.pkl')
else:
    print('SAVE=False (no artifact overwrite)')


## 7) Key Takeaways

- Multiple modeling approaches were compared, not just one model.
- Threshold tuning is explicit and security-oriented.
- Feature ablation demonstrates why full hybrid features matter.
- This notebook complements `models/train.py` and `models/evaluate.py` as a transparent demo artifact.
